In [8]:
import pandas as pd
import glob
import os

In [9]:
def quantify_vueling_emissions(df):
    def calculate_row(row):
        # 1. Unit Conversion (KM to Nautical Miles)
        dist_nm = row['total_distance_km'] * 0.539957
        ac_type = str(row['aircraft_type_icao']).upper()
        
        # 2. Assign Base Coefficients (Tier 2)
        if 'A321' in ac_type:
            lto_fuel, lto_co2 = 219.0, 689.1
            ccd_a, ccd_b = 384.0, 6.00  # A321 specific CCD
        elif 'A319' in ac_type:
            lto_fuel, lto_co2 = 230.0, 725.0
            ccd_a, ccd_b = 300.0, 4.70  # A319 estimate
        else: # Default to A320
            lto_fuel, lto_co2 = 256.0, 806.3
            ccd_a, ccd_b = 334.0, 5.20
            
        # 3. Apply NEO (New Engine Option) Efficiency Gain (~15%)
        # Vueling uses ICAO codes like A20N or A21N for Neos
        if 'N' in ac_type:
            lto_fuel *= 0.85
            lto_co2 *= 0.85
            ccd_a *= 0.85
            ccd_b *= 0.85
            
        # 4. Final Calculations
        ccd_fuel = ccd_a + (ccd_b * dist_nm)
        ccd_co2 = ccd_fuel * 3.15
        
        return pd.Series({
            'LTO_CO2_kg': round(lto_co2, 2),
            'CCD_CO2_kg': round(ccd_co2, 2),
            'Total_CO2_kg': round(lto_co2 + ccd_co2, 2)
        })

    # Apply the function to the dataframe
    emission_results = df.apply(calculate_row, axis=1)
    return pd.concat([df, emission_results], axis=1)

In [ ]:
all_files = glob.glob("../0. Data/*.pq")

columns = ['callsign','flight_number','scheduled_departure_time_utc','aircraft_type_icao','origin_airport','destination_airport','origin_country','destination_country','duration_hours','total_distance_km','great_circle_distance_km']

for file in all_files:
    print("Processing file", file)
    df = pd.read_parquet(file)
    df = df[df['callsign'].str.contains('VLG')][columns]
    print(df['aircraft_type_icao'].unique())

    # Usage with your data:
    df_emissions = quantify_vueling_emissions(df)
    df_emissions['date'] = pd.to_datetime(df_emissions['scheduled_departure_time_utc']).dt.date
    df_emissions['great_circle_ratio'] = df_emissions['great_circle_distance_km'] / df_emissions['total_distance_km']
    df_emissions[['callsign','flight_number','date','aircraft_type_icao','origin_airport','destination_airport','origin_country','destination_country','duration_hours','total_distance_km','great_circle_distance_km', 'great_circle_ratio','LTO_CO2_kg','CCD_CO2_kg','Total_CO2_kg']]

    df_emissions.to_csv("../0. Data/aux_vueling_emissions.csv", index=False, mode='a')

Processing file ./0. Data/2021-07-05-metadata.pq
['A20N' 'A320' 'A321' 'A319']
Processing file ./0. Data/2021-07-06-metadata.pq
['A320' 'A319' 'A20N' 'A321']
Processing file ./0. Data/2021-07-03-metadata.pq
['A320' 'A20N' 'A321' 'A319']
Processing file ./0. Data/2021-07-04-metadata.pq
['A320' 'A20N' 'A321' 'A319']
Processing file ./0. Data/2021-07-01-metadata.pq
['A320' 'A319' 'A321' 'A20N']
Processing file ./0. Data/2021-07-02-metadata.pq
['A20N' 'A321' 'A320' 'A319']
Processing file ./0. Data/2021-07-07-metadata.pq
['A320' 'A321' 'A20N' 'A319']


In [ ]:
df = pd.read_csv("../0. Data/aux_vueling_emissions.csv")
airports = pd.read_csv("../0. Data/airports.csv")[['iata_code','icao_code']].dropna()
df = df.merge(airports, left_on='origin_airport', right_on='icao_code', how='left').drop(columns=['icao_code']).rename(columns={'iata_code':'origin_icao'})
df = df.merge(airports, left_on='destination_airport', right_on='icao_code', how='left').drop(columns=['icao_code']).rename(columns={'iata_code':'destination_icao'})
df.drop(columns=['origin_airport','destination_airport'], inplace=True)
df = df[['callsign','flight_number','date','aircraft_type_icao','origin_icao','destination_icao','origin_country','destination_country','duration_hours','total_distance_km','great_circle_distance_km', 'great_circle_ratio','LTO_CO2_kg','CCD_CO2_kg','Total_CO2_kg']]
df = df[df['origin_icao'].notna() & df['destination_icao'].notna()]
df.to_csv("./0. Data/vueling_emissions.csv", index=False)
os.remove("./0. Data/aux_vueling_emissions.csv")

In [ ]:
df = pd.read_csv('../0. Data/vueling_emissions.csv')

In [ ]:
# Create summary table grouped by date
summary = df.groupby('date').agg({
    'LTO_CO2_kg': 'sum',
    'CCD_CO2_kg': 'sum',
    'Total_CO2_kg': 'sum',
    'total_distance_km': 'sum',
    'callsign': 'count'
}).rename(columns={'callsign': 'Number_of_Flights'})
summary['LTO_CO2_tn'] = summary['LTO_CO2_kg']/1000
summary['CCD_CO2_tn'] = summary['CCD_CO2_kg']/1000
summary['Total_CO2_tn'] = summary['Total_CO2_kg']/1000
summary = summary[['Number_of_Flights', 'total_distance_km', 'LTO_CO2_tn', 'CCD_CO2_tn', 'Total_CO2_tn']].reset_index()
summary.to_excel('./0. Data/vueling_emissions_summary_raw.xlsx')

In [43]:
summary.iloc[:,1:].mean().rename('1 Week Average per Day')

Number_of_Flights       200.857143
total_distance_km    235367.208571
LTO_CO2_tn              152.087536
CCD_CO2_tn             2225.506384
Total_CO2_tn           2377.594179
Name: 1 Week Average per Day, dtype: float64